# Comparison of models to test explanation of CE signatures

In [1]:
import os
import re
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from datetime import datetime as dt

import warnings; warnings.filterwarnings('ignore')

In [2]:
DATA_LOC = '../data'

# path to processed and ready data
DATA_PATH = os.path.join(DATA_LOC, 'lme_ready')

# path to save data to on completion
VIS_PATH = os.path.join('data', 'web_vis')

# post processing & ready for data visualization steps
FINAL_DOC = os.path.join('data', 'final-document.tsv')

In [3]:
df = pd.read_table(FINAL_DOC, sep='\t')
print(df.shape)
df.head()

(281817, 26)


,x_speaker,x_in_common,x_our_thoughts_synced_up_sr1,x_developed_joint_perspective_sr2,x_thoughts_became_more_alike_sr5,x_saw_world_in_same_way_sr8,y_speaker,y_in_common,y_our_thoughts_synced_up_sr1,y_developed_joint_perspective_sr2,...,turn_delta,nx,ny,Hxy,resid,x_conv_leader,x_responsive_1,x_responsive_2,x_responsive_3,x_you_are_good_listener
0,5e97478f8b670f09dad1fd02,3.0,5.0,5.0,4.0,6.0,5f13bae4a544193e6b94b0cf,5.0,5.0,6.0,...,1,13.0,15.0,2.277085,-0.437443,5.0,5.0,5.0,6.0,7.0
1,5e97478f8b670f09dad1fd02,3.0,5.0,5.0,4.0,6.0,5f13bae4a544193e6b94b0cf,5.0,5.0,6.0,...,3,13.0,16.0,2.241708,-0.460827,5.0,5.0,5.0,6.0,7.0
2,5e97478f8b670f09dad1fd02,3.0,5.0,5.0,4.0,6.0,5f13bae4a544193e6b94b0cf,5.0,5.0,6.0,...,5,13.0,26.0,2.280164,-0.301180,5.0,5.0,5.0,6.0,7.0
3,5e97478f8b670f09dad1fd02,3.0,5.0,5.0,4.0,6.0,5f13bae4a544193e6b94b0cf,5.0,5.0,6.0,...,7,13.0,17.0,1.859337,-0.831346,5.0,5.0,5.0,6.0,7.0
4,5e97478f8b670f09dad1fd02,3.0,5.0,5.0,4.0,6.0,5f13bae4a544193e6b94b0cf,5.0,5.0,6.0,...,9,13.0,6.0,2.359147,-0.465142,5.0,5.0,5.0,6.0,7.0


In [4]:
df['resid2'] = df['resid']**2

In [5]:
df['conversation_id'].nunique()

34

## LME Regression: Predicting CE

In [6]:
import statsmodels.formula.api as smf

In [7]:
test_var = [
    # 'x_conv_leader',
    'x_in_common',
    'x_responsive_1',
    'x_responsive_2',
    'x_responsive_3',
    'x_our_thoughts_synced_up_sr1',
    'x_developed_joint_perspective_sr2',
    'x_thoughts_became_more_alike_sr5',
    'x_saw_world_in_same_way_sr8',
    'x_you_are_good_listener'
]

In [8]:
data = {'params': [], 'aic': []}

In [9]:
for var in tqdm(test_var):
    ######## linear model
    model = "resid ~ x_conv_leader*{}".format(var)
    md = smf.glm(model,data=df)
    # md = smf.ols(model,data=df)
    mdf = md.fit()
    
    # model parameters table
    result = {'var': var}
    for p in mdf.params.index:
        result[p] = '{} ({})'.format(
            np.format_float_scientific(mdf.params[p], precision=2), 
            np.format_float_scientific(mdf.bse[p], precision=2)
        ) + ''.join(['*'] * (int(mdf.pvalues[p] < .05) + int(mdf.pvalues[p] < .01) + int(mdf.pvalues[p] < .00001) ))
    # result['r2'] = mdf.rsquared
    data['params'] += [result]
    
    # aic table
    aic_result = {
        'var': var, 
        'linear': (2*len(mdf.params)) - (2 * mdf.llf)
    }
    
    ######## square model
    model = "resid2 ~ x_conv_leader*{}".format(var)
    md = smf.glm(model,data=df)
    # md = smf.ols(model,data=df)
    mdf = md.fit()
    
    # model parameters table
    result = {'var': var}
    for p in mdf.params.index:
        result[p] = '{} ({})'.format(
            np.format_float_scientific(mdf.params[p], precision=2), 
            np.format_float_scientific(mdf.bse[p], precision=2)
        ) + ''.join(['*'] * (int(mdf.pvalues[p] < .05) + int(mdf.pvalues[p] < .01) + int(mdf.pvalues[p] < .00001) ))
    # result['r2'] = mdf.rsquared
    data['params'] += [result]
    
    aic_result['squared'] = (2*len(mdf.params)) - (2 * mdf.llf)
    
    data['aic'] += [aic_result]
    
data['params'] = pd.DataFrame(data['params'])
data['aic'] = pd.DataFrame(data['aic'])

  0%|          | 0/9 [00:00<?, ?it/s]

### Looking at models and parameters

In [10]:
data['params'] = data['params'].transpose()
data['params']['var'] = data['params'].index
data['params'] = data['params'][['var'] + [col for col in data['params'].columns if str(col) != 'var']]

In [11]:
data['params'].head()

,var,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17
var,var,x_in_common,x_in_common,x_responsive_1,x_responsive_1,x_responsive_2,x_responsive_2,x_responsive_3,x_responsive_3,x_our_thoughts_synced_up_sr1,x_our_thoughts_synced_up_sr1,x_developed_joint_perspective_sr2,x_developed_joint_perspective_sr2,x_thoughts_became_more_alike_sr5,x_thoughts_became_more_alike_sr5,x_saw_world_in_same_way_sr8,x_saw_world_in_same_way_sr8,x_you_are_good_listener,x_you_are_good_listener
Intercept,Intercept,-1.77e-03 (1.51e-02),4.22e-01 (1.37e-01)**,-2.26e-04 (2.83e-02),2.15e+00 (2.57e-01)***,-1.61e-03 (3.38e-02),2.91e+00 (3.07e-01)***,-9.85e-04 (3.77e-02),1.78e+00 (3.42e-01)***,-2.77e-03 (1.58e-02),2.09e+00 (1.43e-01)***,-2.26e-03 (1.72e-02),3.4e+00 (1.56e-01)***,-2.39e-03 (1.44e-02),2.30e+00 (1.30e-01)***,-1.78e-03 (1.48e-02),1.98e+00 (1.34e-01)***,-1.28e-03 (4.40e-02),4.08e+00 (3.99e-01)***
x_conv_leader,x_conv_leader,1.05e-05 (3.28e-03),1.99e-01 (2.98e-02)***,-2.99e-04 (5.39e-03),-6.01e-03 (4.89e-02),-6.63e-05 (6.04e-03),-1.85e-02 (5.48e-02),-1.20e-04 (6.27e-03),1.70e-01 (5.69e-02)**,2.59e-04 (3.42e-03),-8.68e-02 (3.11e-02)**,9.24e-05 (3.97e-03),-2.92e-01 (3.60e-02)***,1.21e-04 (3.17e-03),-4.37e-02 (2.88e-02),3.83e-05 (3.25e-03),4.23e-02 (2.95e-02),-1.36e-04 (7.17e-03),-1.46e-01 (6.50e-02)*
x_in_common,x_in_common,-3.14e-05 (2.74e-03),4.24e-01 (2.49e-02)***,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
x_conv_leader:x_in_common,x_conv_leader:x_in_common,3.86e-06 (5.88e-04),-6.67e-02 (5.33e-03)***,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
search_cols = np.array([col for col in np.unique(data['params'].values[0,:]) if col != 'var'])
search_cols

array(['x_developed_joint_perspective_sr2', 'x_in_common',
       'x_our_thoughts_synced_up_sr1', 'x_responsive_1', 'x_responsive_2',
       'x_responsive_3', 'x_saw_world_in_same_way_sr8',
       'x_thoughts_became_more_alike_sr5', 'x_you_are_good_listener'],
      dtype='<U33')

In [13]:
d = []

In [14]:
for col in search_cols:
    sel = (data['params'].values[0,:] == col).nonzero()[0]-1
    vs = data['params'][['var']+sel.tolist()].values
    d += [vs]
d = np.concatenate(d,axis=0)
d = pd.DataFrame(d, columns=['var', '(1)', '(2)'])

In [15]:
sel = d['var'].isin(['var'])
d['var'].loc[sel] = d['(1)'].loc[sel]

In [16]:
data['params']=d

In [17]:
data['params'].head(n=100)

,var,(1),(2)
0,x_developed_joint_perspective_sr2,x_developed_joint_perspective_sr2,x_developed_joint_perspective_sr2
1,Intercept,-2.26e-03 (1.72e-02),3.4e+00 (1.56e-01)***
2,x_conv_leader,9.24e-05 (3.97e-03),-2.92e-01 (3.60e-02)***
3,x_in_common,NaN,NaN
4,x_conv_leader:x_in_common,NaN,NaN
...,...,...,...
95,x_our_thoughts_synced_up_sr1,NaN,NaN
96,x_conv_leader:x_our_thoughts_synced_up_sr1,NaN,NaN
97,x_developed_joint_perspective_sr2,NaN,NaN
98,x_conv_leader:x_developed_joint_perspective_sr2,NaN,NaN


In [18]:
fin_data_path = os.path.join('data', 'parameter-estimates-reversed.csv')

In [19]:
data['params'].to_csv(fin_data_path,  header=False, encoding='utf-8')

In [20]:
with open(fin_data_path.replace('.csv', '.txt'), 'w') as f:
    txt =  data['params'].to_latex(index=False).replace('\\toprule', '\\hline').replace('\\midrule', '\\hline\\hline').replace('\\bottomrule', '\\hline')

    txt = txt.replace('\\\\', '\\\\\hline').replace('|', '$|$').replace('_delta', ' $\Delta$')
    f.write(txt)
    f.close()

### Comparing AIC scores

In [21]:
aic_data_path = os.path.join('data', 'aic-reversed.csv')

In [22]:
data['aic']['model_sel'] = data['aic']['linear'] - data['aic']['squared']
data['aic'].head(n=100)

,var,linear,squared,model_sel
0,x_in_common,990228.822867,2.232773e+06,-1.242545e+06
1,x_responsive_1,990228.818882,2.233086e+06,-1.242857e+06
2,x_responsive_2,990228.822698,2.233048e+06,-1.242819e+06
3,x_responsive_3,990228.822391,2.233041e+06,-1.242812e+06
4,x_our_thoughts_synced_up_sr1,990228.817926,2.233073e+06,-1.242844e+06
5,x_developed_joint_perspective_sr2,990228.822651,2.233061e+06,-1.242832e+06
6,x_thoughts_became_more_alike_sr5,990228.821801,2.233082e+06,-1.242853e+06
7,x_saw_world_in_same_way_sr8,990228.822447,2.233053e+06,-1.242824e+06
8,x_you_are_good_listener,990228.822284,2.233017e+06,-1.242788e+06


In [32]:
data['aic'].to_csv(aic_data_path, index=False, encoding='utf-8')

In [ ]:
with open(aic_data_path.replace('.csv', '.txt'), 'w') as f:
    txt =  data['aic'].to_latex(index=False).replace('\\toprule', '\\hline').replace('\\midrule', '\\hline\\hline').replace('\\bottomrule', '\\hline')

    txt = txt.replace('\\\\', '\\\\\hline').replace('|', '$|$').replace('_delta', ' $\Delta$')
    f.write(txt)
    f.close()